**Context.** This title cell identifies the notebook as the Problem 2 solution. The notebook is organized around the tiny Shakespeare baseline comparison, hyperparameter study, and sequence-length-50 extension.


# ECGR 4106 Homework 2 - Problem 2

**Context.** This preflight cell checks whether PyTorch imports successfully in Colab and repairs the runtime if Colab's PyTorch installation is corrupted.


In [1]:
# PyTorch import preflight for Colab
# If Colab's preinstalled PyTorch is in a broken state, this cell repairs it and restarts the runtime.
import importlib
import os
import subprocess
import sys

def ensure_torch_imports():
    try:
        return importlib.import_module("torch")
    except RuntimeError as exc:
        message = str(exc)
        known_colab_torch_import_bug = "THPDtypeType.tp_dict" in message or "Dtype.cpp" in message
        if not known_colab_torch_import_bug:
            raise
        print("PyTorch failed during import because the current runtime has a broken torch install.")
        print("Reinstalling PyTorch now. The runtime will restart automatically after installation.")
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--quiet",
                "--upgrade",
                "--force-reinstall",
                "torch",
                "numpy",
                "pandas",
                "matplotlib",
            ]
        )
        os.kill(os.getpid(), 9)

torch_check = ensure_torch_imports()
print(f"PyTorch import check passed: torch {torch_check.__version__}")


PyTorch import check passed: torch 2.11.0+cu128


**Context.** This setup cell imports the required packages, fixes the random seed, selects the GPU when available, and creates a folder for saved result tables.


In [2]:
# Colab-friendly setup
import math
import os
import random
import time
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

OUTPUT_DIR = Path("/content/ECGR4106_HW2_outputs") if Path("/content").exists() else Path("ECGR4106_HW2_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


Using device: cuda


**Context.** The shared-code section defines the common dataset, vocabulary, model class, complexity estimates, training loop, evaluation metrics, and generation helper.


## Shared Model, Dataset, Metrics, and Training Code

**Context.** This code cell implements the reusable character-level language modeling pipeline used for all Problem 2 experiments. Keeping one pipeline makes the LSTM and GRU comparisons fair because they share the same data split, optimizer, metrics, and timing code.


In [3]:
class CharWindowDataset(Dataset):
    def __init__(self, encoded_text, seq_len):
        self.data = torch.tensor(encoded_text, dtype=torch.long)
        self.seq_len = int(seq_len)
        if len(self.data) <= self.seq_len:
            raise ValueError("Text is too short for the requested sequence length.")

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx : idx + self.seq_len]
        y = self.data[idx + 1 : idx + self.seq_len + 1]
        return x, y


class CharVocabulary:
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}

    def encode(self, text):
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.itos[int(i)] for i in ids)

    def __len__(self):
        return len(self.chars)


def make_loaders(text, seq_len, batch_size=64, val_fraction=0.2):
    vocab = CharVocabulary(text)
    encoded = vocab.encode(text)
    split = max(seq_len + 1, int(len(encoded) * (1 - val_fraction)))
    train_encoded = encoded[:split]
    val_encoded = encoded[split - seq_len :]
    train_ds = CharWindowDataset(train_encoded, seq_len)
    val_ds = CharWindowDataset(val_encoded, seq_len)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, vocab


class CharLanguageModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        rnn_type="LSTM",
        embed_dim=64,
        hidden_size=128,
        num_layers=1,
        fc_hidden=0,
        dropout=0.0,
    ):
        super().__init__()
        self.rnn_type = rnn_type.upper()
        self.embed_dim = int(embed_dim)
        self.hidden_size = int(hidden_size)
        self.num_layers = int(num_layers)
        self.fc_hidden = int(fc_hidden)
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[self.rnn_type]
        recurrent_dropout = dropout if num_layers > 1 else 0.0
        self.recurrent = rnn_cls(
            input_size=embed_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )
        if fc_hidden and fc_hidden > 0:
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_size, fc_hidden),
                nn.ReLU(),
                nn.Linear(fc_hidden, vocab_size),
            )
        else:
            self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, vocab_size))

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.recurrent(x, hidden)
        logits = self.head(out)
        return logits, hidden


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def model_size_mb(model):
    total_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    total_bytes += sum(b.numel() * b.element_size() for b in model.buffers())
    return total_bytes / (1024 ** 2)


def recurrent_gate_count(rnn_type):
    return {"RNN": 1, "GRU": 3, "LSTM": 4}[rnn_type.upper()]


def approximate_recurrent_madds_per_sequence(rnn_type, seq_len, embed_dim, hidden_size, num_layers, vocab_size, fc_hidden=0):
    # Approximate multiply-add terms for one sample sequence. Biases and activations are omitted.
    gates = recurrent_gate_count(rnn_type)
    total = 0
    for layer in range(num_layers):
        input_dim = embed_dim if layer == 0 else hidden_size
        total += seq_len * gates * (input_dim * hidden_size + hidden_size * hidden_size)
    if fc_hidden and fc_hidden > 0:
        total += seq_len * (hidden_size * fc_hidden + fc_hidden * vocab_size)
    else:
        total += seq_len * hidden_size * vocab_size
    return int(total)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(device)
        y = y.to(device)
        logits, _ = model(x)
        loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
        total_loss += loss.item() * y.numel()
        pred = logits.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total += y.numel()
    avg_loss = total_loss / total
    acc = correct / total
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")
    return avg_loss, acc, ppl


def train_one_experiment(
    text,
    seq_len,
    rnn_type,
    *,
    epochs=10,
    batch_size=64,
    embed_dim=64,
    hidden_size=128,
    num_layers=1,
    fc_hidden=0,
    dropout=0.0,
    lr=0.003,
    val_fraction=0.2,
    label=None,
):
    train_loader, val_loader, vocab = make_loaders(text, seq_len, batch_size=batch_size, val_fraction=val_fraction)
    model = CharLanguageModel(
        len(vocab),
        rnn_type=rnn_type,
        embed_dim=embed_dim,
        hidden_size=hidden_size,
        num_layers=num_layers,
        fc_hidden=fc_hidden,
        dropout=dropout,
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = []
    start = time.perf_counter()
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        seen = 0
        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits, _ = model(x)
            loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            running_loss += loss.item() * y.numel()
            seen += y.numel()
        train_loss = running_loss / seen
        val_loss, val_acc, val_ppl = evaluate(model, val_loader, criterion)
        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
                "val_perplexity": val_ppl,
            }
        )
        print(
            f"{label or rnn_type} | seq={seq_len} | epoch {epoch:02d}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}"
        )

    train_time = time.perf_counter() - start
    inference_start = time.perf_counter()
    evaluate(model, val_loader, criterion)
    inference_time = time.perf_counter() - inference_start
    final = history[-1]
    result = {
        "label": label or f"{rnn_type}_seq{seq_len}",
        "model": rnn_type.upper(),
        "seq_len": seq_len,
        "epochs": epochs,
        "embed_dim": embed_dim,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "fc_hidden": fc_hidden,
        "dropout": dropout,
        "train_loss": final["train_loss"],
        "val_loss": final["val_loss"],
        "val_accuracy": final["val_accuracy"],
        "val_perplexity": final["val_perplexity"],
        "training_time_sec": train_time,
        "inference_time_sec": inference_time,
        "parameters": count_parameters(model),
        "model_size_mb": model_size_mb(model),
        "approx_madds_per_sequence": approximate_recurrent_madds_per_sequence(
            rnn_type, seq_len, embed_dim, hidden_size, num_layers, len(vocab), fc_hidden
        ),
        "vocab_size": len(vocab),
        "history": history,
    }
    return model, vocab, result


@torch.no_grad()
def generate_text(model, vocab, prompt, length=300, temperature=0.8):
    model.eval()
    prompt = "".join(ch for ch in prompt if ch in vocab.stoi)
    if not prompt:
        prompt = vocab.chars[0]
    ids = torch.tensor([[vocab.stoi[ch] for ch in prompt]], dtype=torch.long, device=device)
    output = list(prompt)
    logits, hidden = model(ids)
    for _ in range(length):
        logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        output.append(vocab.itos[int(next_id.item())])
        logits, hidden = model(next_id, hidden)
    return "".join(output)


def plot_history(history, title):
    df = pd.DataFrame(history)
    plt.figure(figsize=(7, 4))
    plt.plot(df["epoch"], df["train_loss"], label="train loss")
    plt.plot(df["epoch"], df["val_loss"], label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Cross-entropy loss")
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


**Context.** This section loads the tiny Shakespeare dataset used for the language-model comparison.


## Tiny Shakespeare Data

**Context.** This data cell downloads tiny Shakespeare, reports the full dataset size and vocabulary size, and uses a practical subset for the Colab run. The subset keeps the full comparison manageable while preserving enough text to compare the models.


In [4]:
TINY_SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
TINY_SHAKESPEARE_PATH = OUTPUT_DIR / "tinyshakespeare.txt"

def load_tiny_shakespeare(path=TINY_SHAKESPEARE_PATH, url=TINY_SHAKESPEARE_URL):
    if not path.exists():
        print("Downloading tiny Shakespeare dataset...")
        urllib.request.urlretrieve(url, path)
    text = path.read_text(encoding="utf-8")
    return text

tiny_text_full = load_tiny_shakespeare()
print(f"Tiny Shakespeare full length: {len(tiny_text_full):,} characters")
print(f"Tiny Shakespeare unique characters: {len(set(tiny_text_full))}")

# A practical Colab default for running all comparisons. Increase this for a more exhaustive final run.
MAX_TINY_SHAKESPEARE_CHARS = 120_000
tiny_text = tiny_text_full[:MAX_TINY_SHAKESPEARE_CHARS]
print(f"Using {len(tiny_text):,} characters for this run")


Tiny Shakespeare full length: 1,115,394 characters
Tiny Shakespeare unique characters: 65
Using 120,000 characters for this run


**Context.** Problem 2.1 compares LSTM and GRU models at sequence lengths 20 and 30.


### Problem 2.1: LSTM and GRU at Sequence Lengths 20 and 30

**Context.** This baseline experiment trains the two recurrent model types under the same hyperparameters. The output table supports the requested comparison of loss, accuracy, training time, inference time, parameter count, model size, and computational complexity.


In [5]:
P2_EPOCHS = 5
P2_BATCH_SIZE = 128
P2_EMBED_DIM = 96
P2_HIDDEN_SIZE = 192
P2_NUM_LAYERS = 2
P2_DROPOUT = 0.2
P2_LR = 0.002

p2_part1_results = []
p2_part1_models = {}
for seq_len in [20, 30]:
    for rnn_type in ["LSTM", "GRU"]:
        label = f"P2_base_{rnn_type}_seq{seq_len}"
        model, vocab, result = train_one_experiment(
            tiny_text,
            seq_len,
            rnn_type,
            epochs=P2_EPOCHS,
            batch_size=P2_BATCH_SIZE,
            embed_dim=P2_EMBED_DIM,
            hidden_size=P2_HIDDEN_SIZE,
            num_layers=P2_NUM_LAYERS,
            dropout=P2_DROPOUT,
            lr=P2_LR,
            val_fraction=0.1,
            label=label,
        )
        result["generated_text"] = generate_text(model, vocab, prompt="ROMEO:", length=400, temperature=0.8)
        p2_part1_results.append(result)
        p2_part1_models[label] = (model, vocab)

p2_part1_summary = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in p2_part1_results])
p2_part1_summary.to_csv(OUTPUT_DIR / "problem2_part1_summary.csv", index=False)
p2_part1_summary[
    [
        "model",
        "seq_len",
        "train_loss",
        "val_loss",
        "val_accuracy",
        "val_perplexity",
        "training_time_sec",
        "inference_time_sec",
        "parameters",
        "model_size_mb",
        "approx_madds_per_sequence",
    ]
]


P2_base_LSTM_seq20 | seq=20 | epoch 01/5 | train_loss=1.8196 | val_loss=1.5883 | val_acc=0.5210
P2_base_LSTM_seq20 | seq=20 | epoch 02/5 | train_loss=1.4261 | val_loss=1.5725 | val_acc=0.5264
P2_base_LSTM_seq20 | seq=20 | epoch 03/5 | train_loss=1.3073 | val_loss=1.6070 | val_acc=0.5280
P2_base_LSTM_seq20 | seq=20 | epoch 04/5 | train_loss=1.2337 | val_loss=1.6458 | val_acc=0.5293
P2_base_LSTM_seq20 | seq=20 | epoch 05/5 | train_loss=1.1821 | val_loss=1.6937 | val_acc=0.5247
P2_base_GRU_seq20 | seq=20 | epoch 01/5 | train_loss=1.7275 | val_loss=1.5876 | val_acc=0.5236
P2_base_GRU_seq20 | seq=20 | epoch 02/5 | train_loss=1.4136 | val_loss=1.5974 | val_acc=0.5271
P2_base_GRU_seq20 | seq=20 | epoch 03/5 | train_loss=1.3315 | val_loss=1.6186 | val_acc=0.5283
P2_base_GRU_seq20 | seq=20 | epoch 04/5 | train_loss=1.2887 | val_loss=1.6408 | val_acc=0.5244
P2_base_GRU_seq20 | seq=20 | epoch 05/5 | train_loss=1.2611 | val_loss=1.6539 | val_acc=0.5232
P2_base_LSTM_seq30 | seq=30 | epoch 01/5 | tr

,model,seq_len,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence
0,LSTM,20,1.182062,1.693735,0.524692,5.439758,75.628580,1.277739,536797,2.047718,10556160
1,GRU,20,1.261140,1.653911,0.523154,5.227382,53.313007,0.849102,407005,1.552601,7975680
2,LSTM,30,1.029702,1.824402,0.524481,6.199085,105.064947,1.820556,536797,2.047718,15834240
3,GRU,30,1.117115,1.700892,0.525694,5.478831,73.453637,1.214505,407005,1.552601,11963520


**Context.** Problem 2.2 changes model capacity through the fully connected head, number of recurrent layers, hidden-state size, embedding size, and dropout.


### Problem 2.2: Hyperparameter Comparison

**Context.** This hyperparameter experiment tests small, medium, medium-with-FC, and large configurations for both LSTM and GRU. These results show how extra capacity affects accuracy, runtime, perplexity, and generated text.


In [6]:
hyperparameter_configs = [
    {
        "label": "small_1layer_no_fc",
        "embed_dim": 64,
        "hidden_size": 128,
        "num_layers": 1,
        "fc_hidden": 0,
        "dropout": 0.0,
    },
    {
        "label": "medium_2layer_no_fc",
        "embed_dim": 96,
        "hidden_size": 192,
        "num_layers": 2,
        "fc_hidden": 0,
        "dropout": 0.2,
    },
    {
        "label": "medium_2layer_fc",
        "embed_dim": 96,
        "hidden_size": 192,
        "num_layers": 2,
        "fc_hidden": 128,
        "dropout": 0.2,
    },
    {
        "label": "large_3layer_fc",
        "embed_dim": 128,
        "hidden_size": 256,
        "num_layers": 3,
        "fc_hidden": 256,
        "dropout": 0.3,
    },
]

P2_HYPERPARAM_EPOCHS = 4
P2_HYPERPARAM_SEQ_LEN = 30

p2_hparam_results = []
p2_hparam_models = {}
for cfg in hyperparameter_configs:
    for rnn_type in ["LSTM", "GRU"]:
        label = f"P2_hparam_{rnn_type}_{cfg['label']}"
        model, vocab, result = train_one_experiment(
            tiny_text,
            P2_HYPERPARAM_SEQ_LEN,
            rnn_type,
            epochs=P2_HYPERPARAM_EPOCHS,
            batch_size=P2_BATCH_SIZE,
            embed_dim=cfg["embed_dim"],
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            fc_hidden=cfg["fc_hidden"],
            dropout=cfg["dropout"],
            lr=P2_LR,
            val_fraction=0.1,
            label=label,
        )
        result["generated_text"] = generate_text(model, vocab, prompt="ROMEO:", length=400, temperature=0.8)
        p2_hparam_results.append(result)
        p2_hparam_models[label] = (model, vocab)

p2_hparam_summary = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in p2_hparam_results])
p2_hparam_summary.to_csv(OUTPUT_DIR / "problem2_hyperparameter_summary.csv", index=False)
p2_hparam_summary[
    [
        "label",
        "model",
        "seq_len",
        "embed_dim",
        "hidden_size",
        "num_layers",
        "fc_hidden",
        "dropout",
        "train_loss",
        "val_loss",
        "val_accuracy",
        "val_perplexity",
        "training_time_sec",
        "inference_time_sec",
        "parameters",
        "model_size_mb",
        "approx_madds_per_sequence",
    ]
]


P2_hparam_LSTM_small_1layer_no_fc | seq=30 | epoch 01/4 | train_loss=1.8243 | val_loss=1.6272 | val_acc=0.5149
P2_hparam_LSTM_small_1layer_no_fc | seq=30 | epoch 02/4 | train_loss=1.4287 | val_loss=1.6008 | val_acc=0.5210
P2_hparam_LSTM_small_1layer_no_fc | seq=30 | epoch 03/4 | train_loss=1.3116 | val_loss=1.6260 | val_acc=0.5197
P2_hparam_LSTM_small_1layer_no_fc | seq=30 | epoch 04/4 | train_loss=1.2321 | val_loss=1.6658 | val_acc=0.5184
P2_hparam_GRU_small_1layer_no_fc | seq=30 | epoch 01/4 | train_loss=1.7351 | val_loss=1.6059 | val_acc=0.5203
P2_hparam_GRU_small_1layer_no_fc | seq=30 | epoch 02/4 | train_loss=1.3559 | val_loss=1.6224 | val_acc=0.5249
P2_hparam_GRU_small_1layer_no_fc | seq=30 | epoch 03/4 | train_loss=1.2526 | val_loss=1.6741 | val_acc=0.5192
P2_hparam_GRU_small_1layer_no_fc | seq=30 | epoch 04/4 | train_loss=1.1946 | val_loss=1.7200 | val_acc=0.5120
P2_hparam_LSTM_medium_2layer_no_fc | seq=30 | epoch 01/4 | train_loss=1.7691 | val_loss=1.5466 | val_acc=0.5319
P2_h

,label,model,seq_len,embed_dim,hidden_size,num_layers,fc_hidden,dropout,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence
0,P2_hparam_LSTM_small_1layer_no_fc,LSTM,30,64,128,1,0,0.0,1.232115,1.665809,0.518417,5.289950,13.334280,0.182105,111101,0.423817,3183360
1,P2_hparam_GRU_small_1layer_no_fc,GRU,30,64,128,1,0,0.0,1.194617,1.720024,0.512022,5.584661,12.284917,0.153457,86269,0.329090,2446080
2,P2_hparam_LSTM_medium_2layer_no_fc,LSTM,30,96,192,2,0,0.2,1.084142,1.750462,0.527147,5.757262,84.007265,1.752832,536797,2.047718,15834240
3,P2_hparam_GRU_medium_2layer_no_fc,GRU,30,96,192,2,0,0.2,1.147496,1.685950,0.525864,5.397578,58.741197,1.187527,407005,1.552601,11963520
4,P2_hparam_LSTM_medium_2layer_fc,LSTM,30,96,192,2,128,0.2,1.088724,1.795646,0.526231,6.023365,84.818234,1.778289,557597,2.127064,16454400
5,P2_hparam_GRU_medium_2layer_fc,GRU,30,96,192,2,128,0.2,1.139000,1.713657,0.526586,5.549219,59.963128,1.236286,427805,1.631947,12583680
6,P2_hparam_LSTM_large_3layer_fc,LSTM,30,128,256,3,256,0.3,1.155290,1.704252,0.534447,5.497270,54.953684,0.538155,1537213,5.864002,45688320
7,P2_hparam_GRU_large_3layer_fc,GRU,30,128,256,3,256,0.3,1.196458,1.637408,0.539314,5.141826,45.327485,0.459104,1175229,4.483143,34874880


**Context.** Problem 2.3 increases the sequence length to 50 to test whether a longer context improves validation accuracy enough to justify the added computation.


### Problem 2.3: Sequence Length 50

**Context.** This experiment trains LSTM and GRU models at sequence length 50 and reports the same metrics as the baseline comparison. The multiply-add estimates increase because the recurrent cell is applied at more time steps.


In [7]:
P2_SEQ50_EPOCHS = 5

p2_seq50_results = []
p2_seq50_models = {}
for rnn_type in ["LSTM", "GRU"]:
    label = f"P2_seq50_{rnn_type}"
    model, vocab, result = train_one_experiment(
        tiny_text,
        50,
        rnn_type,
        epochs=P2_SEQ50_EPOCHS,
        batch_size=P2_BATCH_SIZE,
        embed_dim=P2_EMBED_DIM,
        hidden_size=P2_HIDDEN_SIZE,
        num_layers=P2_NUM_LAYERS,
        dropout=P2_DROPOUT,
        lr=P2_LR,
        val_fraction=0.1,
        label=label,
    )
    result["generated_text"] = generate_text(model, vocab, prompt="ROMEO:", length=400, temperature=0.8)
    p2_seq50_results.append(result)
    p2_seq50_models[label] = (model, vocab)

p2_seq50_summary = pd.DataFrame([{k: v for k, v in r.items() if k != "history"} for r in p2_seq50_results])
p2_seq50_summary.to_csv(OUTPUT_DIR / "problem2_seq50_summary.csv", index=False)
p2_seq50_summary[
    [
        "model",
        "seq_len",
        "train_loss",
        "val_loss",
        "val_accuracy",
        "val_perplexity",
        "training_time_sec",
        "inference_time_sec",
        "parameters",
        "model_size_mb",
        "approx_madds_per_sequence",
    ]
]


P2_seq50_LSTM | seq=50 | epoch 01/5 | train_loss=1.6933 | val_loss=1.5327 | val_acc=0.5412
P2_seq50_LSTM | seq=50 | epoch 02/5 | train_loss=1.1807 | val_loss=1.6780 | val_acc=0.5345
P2_seq50_LSTM | seq=50 | epoch 03/5 | train_loss=0.9912 | val_loss=1.8399 | val_acc=0.5231
P2_seq50_LSTM | seq=50 | epoch 04/5 | train_loss=0.8894 | val_loss=1.9754 | val_acc=0.5226
P2_seq50_LSTM | seq=50 | epoch 05/5 | train_loss=0.8274 | val_loss=2.0610 | val_acc=0.5177
P2_seq50_GRU | seq=50 | epoch 01/5 | train_loss=1.5625 | val_loss=1.5636 | val_acc=0.5411
P2_seq50_GRU | seq=50 | epoch 02/5 | train_loss=1.1374 | val_loss=1.6859 | val_acc=0.5363
P2_seq50_GRU | seq=50 | epoch 03/5 | train_loss=1.0271 | val_loss=1.7620 | val_acc=0.5324
P2_seq50_GRU | seq=50 | epoch 04/5 | train_loss=0.9744 | val_loss=1.8024 | val_acc=0.5284
P2_seq50_GRU | seq=50 | epoch 05/5 | train_loss=0.9412 | val_loss=1.8334 | val_acc=0.5282


,model,seq_len,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence
0,LSTM,50,0.827367,2.061026,0.517657,7.854026,164.864044,2.766032,536797,2.047718,26390400
1,GRU,50,0.941207,1.833359,0.528150,6.254864,114.080667,1.819733,407005,1.552601,19939200


**Context.** This section gathers generated output sequences from the trained tiny Shakespeare models.


### Generated Output Sequences

**Context.** This cell prints generated samples. The samples give a qualitative check on whether the models learned Shakespeare-like formatting, speaker labels, word fragments, and line breaks.


In [8]:
def show_generated_samples(results, title):
    print(title)
    print("=" * len(title))
    for result in results:
        print(f"\n--- {result['label']} ---")
        print(result["generated_text"])

show_generated_samples(p2_part1_results, "Problem 2 baseline generated samples")
show_generated_samples(p2_hparam_results, "Problem 2 hyperparameter generated samples")
show_generated_samples(p2_seq50_results, "Problem 2 sequence length 50 generated samples")


Problem 2 baseline generated samples

--- P2_base_LSTM_seq20 ---
ROMEO:Lone word good with a gentleman, present bench'd him cannot
To cure of that's lack'd incensed my life this business. Of or best arms too noble wealls,
And this is strange too noble sound with his faults
That shall be so.

Volsce:
You will be approve all that hundred they have fought,
As we stand no more!
He will the foes a brace of your ears,
Let me good name of my brope.

Citizens:
Against him.



--- P2_base_GRU_seq20 ---
ROMEO:' I they have popue as our own upon
The mother, patricians of themes. Aufidius with strokess, and proding to the fires of dangerous son, that could show the people!

BRUTUS:
Come, come: your warging the more than a grace and thus?

First Sendile:
Let them to do set down before pick.

CORIOLANUS:
You are you our tribunes,
When they say, sir, ho!

MENENIUS:
Not out of a show'd them profess

AUFIDIUS

--- P2_base_LSTM_seq30 ---
ROMEO:NIUS:
Go, get you heard
That would grave me your tentcome to

**Context.** This analysis cell explains the expected tradeoffs among LSTM, GRU, model size, sequence length, runtime, and perplexity.


## Problem 2 Analysis Guide

Use `p2_part1_summary`, `p2_hparam_summary`, and `p2_seq50_summary` for the final written comparison.

Important points to discuss after running the notebook:

- LSTM normally has more parameters and multiply-add work than GRU because it has four gates instead of three.
- GRU is often faster and smaller, while LSTM may perform better when longer dependencies matter.
- More hidden states and more recurrent layers increase parameter count, model size, training time, inference time, and computational complexity.
- A fully connected hidden layer after the recurrent output can improve expressiveness, but it also increases model size and may overfit if the dataset subset or epoch count is small.
- Increasing sequence length to 50 increases the recurrent work per sample. Accuracy can improve if the model benefits from longer context, but training time and memory cost also rise.
- Perplexity is `exp(cross_entropy)`, so lower validation loss means lower validation perplexity and a better language model.


**Context.** The final table cell displays and saves all Problem 2 result summaries so the report can cite exact values.


In [9]:
all_available_tables = {}
for name in [
    "p1_summary",
    "p2_part1_summary",
    "p2_hparam_summary",
    "p2_seq50_summary",
]:
    if name in globals():
        all_available_tables[name] = globals()[name]
        print(f"\n{name}")
        display(globals()[name])

print(f"\nSaved CSV outputs in: {OUTPUT_DIR}")



p2_part1_summary


,label,model,seq_len,epochs,embed_dim,hidden_size,num_layers,fc_hidden,dropout,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence,vocab_size,generated_text
0,P2_base_LSTM_seq20,LSTM,20,5,96,192,2,0,0.2,1.182062,1.693735,0.524692,5.439758,75.628580,1.277739,536797,2.047718,10556160,61,"ROMEO:Lone word good with a gentleman, present..."
1,P2_base_GRU_seq20,GRU,20,5,96,192,2,0,0.2,1.261140,1.653911,0.523154,5.227382,53.313007,0.849102,407005,1.552601,7975680,61,ROMEO:' I they have popue as our own upon\nThe...
2,P2_base_LSTM_seq30,LSTM,30,5,96,192,2,0,0.2,1.029702,1.824402,0.524481,6.199085,105.064947,1.820556,536797,2.047718,15834240,61,"ROMEO:NIUS:\nGo, get you heard\nThat would gra..."
3,P2_base_GRU_seq30,GRU,30,5,96,192,2,0,0.2,1.117115,1.700892,0.525694,5.478831,73.453637,1.214505,407005,1.552601,11963520,61,"ROMEO: sure well met, the nobles so?\n\nCORIOL..."



p2_hparam_summary


,label,model,seq_len,epochs,embed_dim,hidden_size,num_layers,fc_hidden,dropout,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence,vocab_size,generated_text
0,P2_hparam_LSTM_small_1layer_no_fc,LSTM,30,4,64,128,1,0,0.0,1.232115,1.665809,0.518417,5.289950,13.334280,0.182105,111101,0.423817,3183360,61,ROMEO:\nI'll leave they have fought ask the st...
1,P2_hparam_GRU_small_1layer_no_fc,GRU,30,4,64,128,1,0,0.0,1.194617,1.720024,0.512022,5.584661,12.284917,0.153457,86269,0.329090,2446080,61,"ROMEO: Titus Lartius, malice hurbule or rage o..."
2,P2_hparam_LSTM_medium_2layer_no_fc,LSTM,30,4,96,192,2,0,0.2,1.084142,1.750462,0.527147,5.757262,84.007265,1.752832,536797,2.047718,15834240,61,ROMEO:\nHe'll bring the man as a most curses o...
3,P2_hparam_GRU_medium_2layer_no_fc,GRU,30,4,96,192,2,0,0.2,1.147496,1.685950,0.525864,5.397578,58.741197,1.187527,407005,1.552601,11963520,61,ROMEO: See him to and envy you.\n\nCOMINIUS:\n...
4,P2_hparam_LSTM_medium_2layer_fc,LSTM,30,4,96,192,2,128,0.2,1.088724,1.795646,0.526231,6.023365,84.818234,1.778289,557597,2.127064,16454400,61,"ROMEO: hie, my lord.\n\nCOMINIUS:\nWhat is the..."
5,P2_hparam_GRU_medium_2layer_fc,GRU,30,4,96,192,2,128,0.2,1.139000,1.713657,0.526586,5.549219,59.963128,1.236286,427805,1.631947,12583680,61,ROMEO:\nNot unlows to the Capitol be consul.\n...
6,P2_hparam_LSTM_large_3layer_fc,LSTM,30,4,128,256,3,256,0.3,1.155290,1.704252,0.534447,5.497270,54.953684,0.538155,1537213,5.864002,45688320,61,"ROMEO::\nNo, no.\n\nCORIOLANUS:\nO, good madam..."
7,P2_hparam_GRU_large_3layer_fc,GRU,30,4,128,256,3,256,0.3,1.196458,1.637408,0.539314,5.141826,45.327485,0.459104,1175229,4.483143,34874880,61,ROMEO:A:\nI have been none that a soldier: and...



p2_seq50_summary


,label,model,seq_len,epochs,embed_dim,hidden_size,num_layers,fc_hidden,dropout,train_loss,val_loss,val_accuracy,val_perplexity,training_time_sec,inference_time_sec,parameters,model_size_mb,approx_madds_per_sequence,vocab_size,generated_text
0,P2_seq50_LSTM,LSTM,50,5,96,192,2,0,0.2,0.827367,2.061026,0.517657,7.854026,164.864044,2.766032,536797,2.047718,26390400,61,"ROMEO:\nI think, they shall tell you\nA pretty..."
1,P2_seq50_GRU,GRU,50,5,96,192,2,0,0.2,0.941207,1.833359,0.528150,6.254864,114.080667,1.819733,407005,1.552601,19939200,61,ROMEO:\nIn broin'd than a Roman.\n\nThird Citi...



Saved CSV outputs in: /content/ECGR4106_HW2_outputs
